### 1. Visualize the 1D spectrum from default HST reduction
NOTE: for the actual research work we will NOT be using the default reduction. Because that uses default params and no bg extraction (i think but maybe check idk). We're also doing bg removal and cosmic ray rejection and etc. and MAST traces bg regions might be totally wrong.

In [ ]:
x1d_230_1 = Table.read(Path('./Data/2024tvi/HST/of8b05010/sp230_1_x1d.fits'), hdu=1)

dq512_idx_1 = (x1d_230_1['DQ'] & 512 != 512)[0]
dq16_idx_1 = (x1d_230_1['DQ'] & 16 != 16)[0]

dq512_idx_2 = (x1d_230_2['DQ'] & 512 != 512)[0]
dq16_idx_2 = (x1d_230_2['DQ'] & 16 != 16)[0]

dq512_idx_430 = (x1d_430['DQ'] & 512 != 512)[0]
dq16_idx_430 = (x1d_430['DQ'] & 16 != 16)[0]

dq512_idx_750 = (x1d_750['DQ'] & 512 != 512)[0]
dq16_idx_750 = (x1d_750['DQ'] & 16 != 16)[0]

wvl_230_all, flx_230_all, err_230_all = np.concatenate((wvl_230_1[dq16_idx_1 & dq512_idx_1], wvl_230_2[dq16_idx_2 & dq512_idx_2])), \
    np.concatenate((flux_230_1[dq16_idx_1 & dq512_idx_1], flux_230_2[dq16_idx_2 & dq512_idx_2])), \
    np.concatenate((err_230_1[dq16_idx_1 & dq512_idx_1], err_230_2[dq16_idx_2 & dq512_idx_2]))

### 2. Manual Cosmic Ray Rejection

In [ ]:
stistools.ocrreject.ocrreject(
    './Data/2024iss/HST/of8b02010/of8b02010_flt.fits', 
    "./Data/2024iss/HST/of8b02010/230_1_crj.fits", 
    verbose=True, 
    trailer="230_1_cr.trl"
)

### 3. Re-extracting the Spectrum

In [ ]:
stistools.x1d.x1d(
    './Data/2024tvi/HST/of8b05010/230_1_crj.fits', 
    output="./Data/2024tvi/HST/of8b05010/sp230_1_x1d_crj.fits", 
    verbose=True, 
    trailer="sp230_1_cr.trl", 
    extrsize=2, 
    bk1offst=-5, 
    bk2offst=5
)

# NOTE (IMPORTANT): in the .x1d.x1d u can manually enter the center (not shown in code)
# NOTE: every time u make an x1d file, u gotta delete it to run again (cant override it)

# fuller example showing the manual center: maxsrch=0 means dont search, just use a2center. tight bg straddling the trace
stistools.x1d.x1d("../Data/ZTF25ABVBCZT/HST/ofgd05050/ofgd05050_crj.fits",
                  output="../Data/ZTF25ABVBCZT/HST/ofgd05050/430_x1d-ext_1.fits",
                  verbose=True, trailer="../Data/ZTF25ABVBCZT/HST/ofgd05050/ofgd05050_trl",
                  extrsize=5, maxsrch=0, a2center=893, bk1offst=-8, bk2offst=8, bk1size=10, bk2size=10)


### 4. Helper Function: Show Extraction Regions
Function to show extraction regions for a given `x1d` file.

In [ ]:
def show_extraction_regions(x1d_filename, flt_filename, sci_ext=1, row=0, xrange=None, yrange=None):
    fig, axes = plt.subplots(1, 2, sharey=True)
    fig.tight_layout()
    fig.subplots_adjust(wspace=0.2, hspace=0.2, top=0.88)
    fig.set_figwidth(10)
    fig.set_figheight(5)
    fig.suptitle(os.path.basename(x1d_filename))
    
    x1d = fits.getdata(x1d_filename, ext=sci_ext)[row]
    flt = fits.getdata(flt_filename, ext=('SCI', sci_ext))
    
    # LEFT & RIGHT PLOTS:
    for ax in axes[0:2]:
        # Display the 2D FLT spectrum on the left:
        ax.imshow(flt, origin='lower', interpolation='none', aspect='auto', vmin=-6, vmax=15)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')

    # RIGHT PLOT SPECIFICS:
    axes[1].set_title(f"A2CENTER={x1d['A2CENTER']:.2f}")
    
    # Extraction region in red:
    axes[1].plot(np.arange(1024), x1d['EXTRLOCY'] - 1, 'r:', alpha=0.6)
    axes[1].plot(np.arange(1024), x1d['EXTRLOCY'] - 1 - x1d['EXTRSIZE'] // 2, color='red', alpha=0.6)
    axes[1].plot(np.arange(1024), x1d['EXTRLOCY'] - 1 + x1d['EXTRSIZE'] // 2, color='red', alpha=0.6)
    
    # Background regions in orange:
    axes[1].plot(np.arange(1024), x1d['EXTRLOCY'] - 1 + x1d['BK1OFFST'] - x1d['BK1SIZE'] // 2, color='orange', alpha=0.6)
    axes[1].plot(np.arange(1024), x1d['EXTRLOCY'] - 1 + x1d['BK1OFFST'] + x1d['BK1SIZE'] // 2, color='orange', alpha=0.6)
    axes[1].plot(np.arange(1024), x1d['EXTRLOCY'] - 1 + x1d['BK2OFFST'] - x1d['BK2SIZE'] // 2, color='orange', alpha=0.6)
    axes[1].plot(np.arange(1024), x1d['EXTRLOCY'] - 1 + x1d['BK2OFFST'] + x1d['BK2SIZE'] // 2, color='orange', alpha=0.6)
    
    axes[0].set_xlim(-0.5, 1023.5)
    axes[0].set_ylim(-0.5, 1023.5)
    axes[1].set_xlim(-0.5, 1023.5)
    axes[1].set_ylim(-0.5, 1023.5)
    
    if xrange is not None:
        axes[0].set_xlim(xrange[0], xrange[1])
        axes[1].set_xlim(xrange[0], xrange[1])
    if yrange is not None:
        axes[0].set_ylim(yrange[0], yrange[1])
        axes[1].set_ylim(yrange[0], yrange[1])

### 5. Visualize 2D Spectrum, Background Regions, and Aperture Choice

In [ ]:
f1d_230_1 = Path('./Data/2024iss/HST/of8b02010/230_x1d.fits')
f1d_230_1_CR = Path('./Data/2024iss/HST/of8b02010/sp230_1_x1d_crj.fits')
flt_230_1 = Path('./Data/2024iss/HST/of8b02010/of8b02010_flt.fits')

show_extraction_regions(f1d_230_1_CR, flt_230_1, yrange=[870, 920])
show_extraction_regions(f1d_230_1, flt_230_1, yrange=[870, 920])